# Perturbation prediction for unseen cell types (GSE274113)

The Perturb-multiome data of GSE274113 hold paired RNA and ATAC of human
haematopoietic cells with transcription-factor knockouts. This notebook repeats the
leave-one-cell-type-out test of Fig. 3a of the paper: for one cell type at a time,
all its perturbed cells are removed from training (its control cells stay),
MultiFlow is trained on the rest, and the response of the held-out cell type to
every knockout is predicted and compared with the observed cells.

| Step | Function |
|---|---|
| Download the data | `mf.download_dataset` |
| Train the paired autoencoder and encode | `mf.train_autoencoder`, `mf.encode` |
| Hold out one cell type | `mf.prepare_perturbation_fold` |
| Train MultiFlow | `mf.train` |
| Predict every knockout | `mf.predict_perturbations` |
| Score against the observed cells | `mf.metrics.evaluate_perturbation` |

Everything is trained from scratch, and the only requirement is
`pip install multiflow-omics`. The full run trains one autoencoder and one flow per
held-out cell type, so plan for a GPU and several hours; `CELL_TYPES` below limits
the run to a few cell types. To check the installation first, set `QUICK = True`;
the same steps then run on a small simulated dataset in about a minute on a CPU.

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd

import multiflow_omics as mf

QUICK = os.environ.get("MULTIFLOW_QUICK") == "1"  # or simply: QUICK = True
DEVICE = None  # None uses the GPU when there is one
CELL_TYPES = None  # None holds out every cell type in turn; or a list of names

if QUICK:  # simulated data and small networks: about a minute on a CPU
    WORK = Path("runs/perturbation_quick")
    AUTOENCODER = dict(epochs=40, rna_dims=(128, 64, 16), atac_dims=(128, 64, 16))
    FLOW = dict(epochs=15, hidden_dim=64, batch_size=64)
else:  # the settings of the paper
    WORK = Path("runs/perturbation")
    AUTOENCODER = dict(epochs=100)
    FLOW = dict(epochs=600)

started = time.time()
print("multiflow-omics", mf.__version__)

## 1. Data

The processed file is 31 GB (Zenodo). RNA holds raw counts and ATAC is binary;
`cell_type` and `perturbation` annotate every cell, and unperturbed cells are
labelled `control`. The table lists the number of cells per cell type and
perturbation.

In [ ]:
if QUICK:
    data = WORK / "simulated.h5mu"
    if not data.exists():
        mf.simulate_h5mu(data)
else:
    data = mf.download_dataset(
        "gse274113", "data/GSE274113_filtered.h5mu", accept_license=True
    )

obs = mf.read_obs(data)
cells = (
    obs.astype({"cell_type": str, "perturbation": str})
    .groupby(["cell_type", "perturbation"])
    .size()
    .unstack(fill_value=0)
)
print(f"{len(obs):,} cells, {cells.shape[0]} cell types, {cells.shape[1] - 1} knockouts")
cells

## 2. Autoencoder

For perturbation prediction both modalities go through one paired autoencoder: RNA
with a negative-binomial decoder on raw counts, ATAC with a Bernoulli decoder. As in
the paper, it is trained once on all cells; the hold-out applies to the flow.

In [ ]:
autoencoder = WORK / "autoencoder" / "autoencoder.pt"
if not autoencoder.exists():
    mf.train_autoencoder(data, autoencoder.parent, device=DEVICE, **AUTOENCODER)

encoded = WORK / "encoded.h5mu"
if not encoded.exists():
    mf.encode(
        data, encoded, task="perturbation", autoencoder=autoencoder, device=DEVICE
    )

## 3. Leave one cell type out

For each held-out cell type:

1. `prepare_perturbation_fold` removes its perturbed cells and stores the context of
   every cell type (the mean latent state of its training cells);
2. `train` fits MultiFlow, conditioned on the context and on the perturbation;
3. `predict_perturbations` samples as many cells per knockout as were observed,
   applies the training-only mean shift of the paper (the mean effect of the
   knockout in the training cell types is added to the control mean of the held-out
   cell type) and decodes RNA and ATAC;
4. `evaluate_perturbation` compares predicted and observed cells.

The metrics use the 100 genes that respond most strongly in the observed data
(largest absolute t statistic, knockout against control):

- `pcc_delta`: correlation between the predicted and the observed change of the
  mean expression (higher is better);
- `delta_mse`: mean squared difference between the two changes (lower is better);
- `mmd`: distance between predicted and observed cells (lower is better);
- `ilisi`: the scIB iLISI, added when the optional package `scib` is installed.

Every step writes a file and is skipped when that file exists, so an interrupted
run continues where it stopped.

In [ ]:
held_out = CELL_TYPES or [name for name in cells.index if name.lower() != "nan"]
use_ilisi = mf.metrics.scib_available()
tables = []

for cell_type in held_out:
    folder = WORK / "folds" / cell_type.replace("/", "-").replace(" ", "_")
    fold, run = folder / "fold.h5mu", folder / "multiflow"
    predicted, result = folder / "predicted.h5mu", folder / "metrics.csv"

    if not predicted.exists():
        fold_started = time.time()
        if not fold.exists():
            mf.prepare_perturbation_fold(encoded, fold, held_out_cell_type=cell_type)
        if not (run / "model.pt").exists():
            mf.train(
                fold,
                run,
                model="perturbation",
                perturbation_key="perturbation",
                device=DEVICE,
                verbose=False,
                **FLOW,
            )
        observed = cells.loc[cell_type].drop("control")
        mf.predict_perturbations(
            run,
            predicted,
            training_fold=fold,
            held_out_cell_type=cell_type,
            autoencoder=autoencoder,
            n_cells=observed[observed >= 5].to_dict(),
            device=DEVICE,
        )
        fold.unlink()  # as large as the data, and not needed again
        minutes = (time.time() - fold_started) / 60
        print(f"{cell_type}: trained and predicted in {minutes:.1f} min")

    if not result.exists():
        mf.metrics.evaluate_perturbation(
            data, predicted, cell_type=cell_type, ilisi=use_ilisi
        ).to_csv(result, index=False)
    tables.append(pd.read_csv(result))

metrics = pd.concat(tables, ignore_index=True)
metrics.to_csv(WORK / "perturbation_metrics.csv", index=False)
print(f"{len(metrics)} cell type and perturbation pairs")

## 4. Results

Mean over the perturbations of each held-out cell type, and the mean over all
pairs.

In [ ]:
columns = [c for c in ("pcc_delta", "delta_mse", "mmd", "ilisi") if c in metrics]
summary = metrics.groupby("cell_type")[columns].mean()
summary.loc["all pairs"] = metrics[columns].mean()
summary.round(3)

For comparison, the paper reports for MultiFlow a mean top-100 PCC-delta of 0.57 and
a top-100 MMD of 0.034 (Fig. 3a). A new training run does not reproduce these digits
exactly, because the autoencoder and the flows are retrained.

## 5. A look at the pairs (optional)

One point per perturbation; the bar marks the mean of the cell type. Needs
`matplotlib`.

In [ ]:
import importlib.util

import numpy as np

if importlib.util.find_spec("matplotlib"):
    import matplotlib.pyplot as plt

    order = summary.drop("all pairs")["pcc_delta"].sort_values(ascending=False).index
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(max(4.0, 0.7 * len(order) + 1.5), 3.4))
    for position, name in enumerate(order):
        values = metrics.loc[metrics["cell_type"] == name, "pcc_delta"].to_numpy()
        jitter = rng.uniform(-0.18, 0.18, size=values.size)
        x = position + jitter
        ax.scatter(x, values, s=14, color="#2a6f97", alpha=0.7, linewidths=0)
        ax.hlines(values.mean(), position - 0.3, position + 0.3, color="#182f40", lw=2)
    ax.axhline(0, color="0.6", linewidth=0.8)
    ax.set_xticks(range(len(order)), order, rotation=45, ha="right")
    ax.set_ylabel("top-100 PCC-delta")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(WORK / "pcc_delta_by_cell_type.png", dpi=200)
    plt.show()

In [ ]:
print(f"finished in {(time.time() - started) / 3600:.2f} h")